# Ceeria Workflow 분석 노트북

## 워크플로우 구조

```
사용자 질문
    │
    ▼
[classify_and_execute]  ← 질문 분류 + SQL/API 직접 실행
    │
    ▼
[chat_upload_retrieve_documents]  ← 업로드 문서 벡터 검색
    │
    ▼
[conditional_retrieve]  ← skip_rag=False일 때만 RAG 검색
    │
    ▼
[generate_response]  ← 최종 LLM 답변 생성
    │
    ▼
OUTPUT
```

## 질문 분류 흐름

| 단계 | 조건 | 처리 |
|------|------|------|
| 1 | LOT ID 감지 | setmo / slot / 이력 API 호출 |
| 2 | 장비 ID 감지 | 장비 상태/정보 API 호출 |
| 3 | FAB+MODEL 패턴 | 장비 집계 API 호출 |
| 3.5 | 공정 이력 쿼리 | operhis API 호출 |
| 4 | 일반 질문 | RAG 검색 → LLM |

---
**이 노트북은 외부 API를 Mock 데이터로 대체하여 로컬에서 실행 가능합니다.**

## 1. 라이브러리 임포트

In [ ]:
import re
import functools
from typing import List, Optional, Literal, Dict, Any
from collections import defaultdict
from pydantic import BaseModel, Field
from IPython.display import Markdown, display

print("임포트 완료")

## 2. 샘플 데이터 정의

실제 API 대신 사용할 Mock 데이터입니다.

In [ ]:
# ============================================================
# 장비(EQP) 샘플 데이터
# ============================================================

SAMPLE_EQP_M15 = [
    # 메인 장비
    {"EQP_ID": "M15A001", "FAC_ID": "M15", "DET_FAC_ID": "M15A",
     "EQP_MODEL_CD": "LPCVD_A", "EQ_GROUP": "CVD", "VENDOR_NM": "AMAT",
     "MGMT_AREA_ID": "CVD", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Up", "EQP_STAT_CD": "RUN",
     "BAY_NM": "BAY-01", "LAST_EVENT_TM": "2026-05-07 08:00:00",
     "PORT_INFO": "P1l진행중lRun,P2l대기lIdle"},
    {"EQP_ID": "M15A002", "FAC_ID": "M15", "DET_FAC_ID": "M15A",
     "EQP_MODEL_CD": "LPCVD_A", "EQ_GROUP": "CVD", "VENDOR_NM": "AMAT",
     "MGMT_AREA_ID": "CVD", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Down", "EQP_STAT_CD": "PM",
     "BAY_NM": "BAY-01", "LAST_EVENT_TM": "2026-05-06 14:30:00",
     "PORT_INFO": ""},
    {"EQP_ID": "M15B001", "FAC_ID": "M15", "DET_FAC_ID": "M15B",
     "EQP_MODEL_CD": "CMP_B", "EQ_GROUP": "CMP", "VENDOR_NM": "KLA",
     "MGMT_AREA_ID": "CMP", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Up", "EQP_STAT_CD": "RUN",
     "BAY_NM": "BAY-02", "LAST_EVENT_TM": "2026-05-07 09:15:00",
     "PORT_INFO": ""},
    # 챔버
    {"EQP_ID": "M15A001_CH1", "FAC_ID": "M15", "DET_FAC_ID": "M15A",
     "EQP_MODEL_CD": "LPCVD_A", "EQ_GROUP": "CVD", "VENDOR_NM": "AMAT",
     "MGMT_AREA_ID": "CVD", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Up", "EQP_STAT_CD": "RUN",
     "BAY_NM": "BAY-01", "LAST_EVENT_TM": "2026-05-07 08:00:00", "PORT_INFO": ""},
    {"EQP_ID": "M15A001_CH2", "FAC_ID": "M15", "DET_FAC_ID": "M15A",
     "EQP_MODEL_CD": "LPCVD_A", "EQ_GROUP": "CVD", "VENDOR_NM": "AMAT",
     "MGMT_AREA_ID": "CVD", "SECTION_GRP_NM": "M15NAND",
     "MES_STAT_TYP": "Down", "EQP_STAT_CD": "FAULT",
     "BAY_NM": "BAY-01", "LAST_EVENT_TM": "2026-05-06 20:00:00", "PORT_INFO": ""},
]

SAMPLE_EQP_M16 = [
    {"EQP_ID": "M16C001", "FAC_ID": "M16", "DET_FAC_ID": "M16C",
     "EQP_MODEL_CD": "ALD_C", "EQ_GROUP": "ALD", "VENDOR_NM": "TEL",
     "MGMT_AREA_ID": "ALD", "SECTION_GRP_NM": "M16DRAM",
     "MES_STAT_TYP": "Up", "EQP_STAT_CD": "RUN",
     "BAY_NM": "BAY-03", "LAST_EVENT_TM": "2026-05-07 07:00:00", "PORT_INFO": ""},
    {"EQP_ID": "M16C002", "FAC_ID": "M16", "DET_FAC_ID": "M16C",
     "EQP_MODEL_CD": "ALD_C", "EQ_GROUP": "ALD", "VENDOR_NM": "TEL",
     "MGMT_AREA_ID": "ALD", "SECTION_GRP_NM": "M16DRAM",
     "MES_STAT_TYP": "Down", "EQP_STAT_CD": "PM",
     "BAY_NM": "BAY-03", "LAST_EVENT_TM": "2026-05-06 18:00:00", "PORT_INFO": ""},
]

# ============================================================
# LOT 샘플 데이터
# ============================================================

SAMPLE_LOT_DATA = {
    "AB123456": [{"LOT_ID": "AB123456", "PROD_ID": "NAND_128G", "STATUS": "ACTIVE"}],
    "CD789012": [{"LOT_ID": "CD789012", "PROD_ID": "DRAM_16G",  "STATUS": "ACTIVE"}],
}

# ============================================================
# SETMO 샘플 데이터
# ============================================================

SAMPLE_SETMO = [
    {"LOT_ID": "AB123456", "ACT_NM": "SetMonitor",
     "OPER_DESC": "CVD 증착", "ACT_DESC": "두께 측정 필요",
     "MEAS_SLOT_NM": "Slot 1,3,5", "ENGR_USER_NM": "김엔지니어"},
    {"LOT_ID": "AB123456", "ACT_NM": "SetMonitor",
     "OPER_DESC": "CMP 연마", "ACT_DESC": "평탄도 확인",
     "MEAS_SLOT_NM": "Slot 2,4", "ENGR_USER_NM": "이엔지니어"},
    {"LOT_ID": "AB123456", "ACT_NM": "makeOnHold",
     "OPER_DESC": "식각 공정", "ACT_DESC": "이상 감지 시 홀드",
     "MEAS_SLOT_NM": "-", "ENGR_USER_NM": "박엔지니어"},
]

# ============================================================
# Slot 샘플 데이터
# ============================================================

SAMPLE_SLOT = [
    {"POSITION_VAL": 1,  "WF_ID": "WF001", "FAB": "M15",
     "OPER_DESC": "CVD 증착",  "EVENT_TM": "2026-05-07 08:00:00"},
    {"POSITION_VAL": 2,  "WF_ID": "WF002", "FAB": "M15",
     "OPER_DESC": "CVD 증착",  "EVENT_TM": "2026-05-07 08:01:00"},
    {"POSITION_VAL": 3,  "WF_ID": "WF003", "FAB": "M15",
     "OPER_DESC": "CMP 연마",  "EVENT_TM": "2026-05-07 08:02:00"},
    {"POSITION_VAL": 5,  "WF_ID": "WF005", "FAB": "M15",
     "OPER_DESC": "검사",      "EVENT_TM": "2026-05-07 08:03:00"},
    {"POSITION_VAL": 10, "WF_ID": "WF010", "FAB": "M15",
     "OPER_DESC": "식각",      "EVENT_TM": "2026-05-07 07:50:00"},
]

# ============================================================
# LOT 이력 샘플 데이터
# ============================================================

SAMPLE_LOTHIS = [
    {"TIMEKEY": "20260507090000", "EVENT_CD": "TRACK_IN",
     "OPER_ID": "CVD001", "CTN_DESC": "CVD 증착",
     "WF_QTY": 25, "PROD_ID": "NAND_128G"},
    {"TIMEKEY": "20260507080000", "EVENT_CD": "TRACK_OUT",
     "OPER_ID": "PHOTO001", "CTN_DESC": "포토 리소그래피",
     "WF_QTY": 25, "PROD_ID": "NAND_128G"},
    {"TIMEKEY": "20260507060000", "EVENT_CD": "TRACK_IN",
     "OPER_ID": "PHOTO001", "CTN_DESC": "포토 리소그래피",
     "WF_QTY": 25, "PROD_ID": "NAND_128G"},
    {"TIMEKEY": "20260506200000", "EVENT_CD": "TRACK_OUT",
     "OPER_ID": "CMP001", "CTN_DESC": "CMP 연마",
     "WF_QTY": 25, "PROD_ID": "NAND_128G"},
    {"TIMEKEY": "20260506150000", "EVENT_CD": "TRACK_IN",
     "OPER_ID": "CMP001", "CTN_DESC": "CMP 연마",
     "WF_QTY": 25, "PROD_ID": "NAND_128G"},
]

# ============================================================
# 공정 이력(OPERHIS) 샘플 데이터
# ============================================================

SAMPLE_OPERHIS = [
    {"LOT_ID": "AB1001", "OPERATIONDESC": "CVD 증착", "OPERLEVEL": 300,
     "WF_QTY": 25, "CTN_DESC": "CVD 증착",
     "MES_PROC_STAT_CD": "COMPLETE", "LAST_EVENT_TM": "2026-05-07 08:00", "FLOW_ID": "FLOW_A"},
    {"LOT_ID": "AB1002", "OPERATIONDESC": "CVD 증착", "OPERLEVEL": 290,
     "WF_QTY": 24, "CTN_DESC": "CVD 증착",
     "MES_PROC_STAT_CD": "COMPLETE", "LAST_EVENT_TM": "2026-05-06 20:00", "FLOW_ID": "FLOW_A"},
    {"LOT_ID": "AB1001", "OPERATIONDESC": "포토 리소그래피", "OPERLEVEL": 280,
     "WF_QTY": 25, "CTN_DESC": "포토 리소그래피",
     "MES_PROC_STAT_CD": "COMPLETE", "LAST_EVENT_TM": "2026-05-07 06:00", "FLOW_ID": "FLOW_A"},
    {"LOT_ID": "AB1001", "OPERATIONDESC": "CMP 연마", "OPERLEVEL": 270,
     "WF_QTY": 25, "CTN_DESC": "CMP 연마",
     "MES_PROC_STAT_CD": "COMPLETE", "LAST_EVENT_TM": "2026-05-06 18:00", "FLOW_ID": "FLOW_A"},
]

print("샘플 데이터 정의 완료")
print(f"  - EQP (M15): {len(SAMPLE_EQP_M15)}건")
print(f"  - EQP (M16): {len(SAMPLE_EQP_M16)}건")
print(f"  - LOT: {len(SAMPLE_LOT_DATA)}건")
print(f"  - SETMO: {len(SAMPLE_SETMO)}건")
print(f"  - SLOT: {len(SAMPLE_SLOT)}건")
print(f"  - LOTHIS: {len(SAMPLE_LOTHIS)}건")
print(f"  - OPERHIS: {len(SAMPLE_OPERHIS)}건")

## 3. Mock API 함수 정의

외부 Datahub API를 샘플 데이터로 대체합니다.

In [ ]:
# ============================================================
# Mock Datahub API
# ============================================================

def call_lotid(lot_id: str) -> List[dict]:
    """LOT ID 조회 Mock"""
    result = SAMPLE_LOT_DATA.get(lot_id.upper(), [])
    print(f"  [MOCK] call_lotid('{lot_id}') → {len(result)}건")
    return result

def call_setmo_check(lot_id: str) -> List[dict]:
    """SETMO 조회 Mock"""
    result = [d for d in SAMPLE_SETMO if d["LOT_ID"] == lot_id.upper()]
    print(f"  [MOCK] call_setmo_check('{lot_id}') → {len(result)}건")
    return result

def call_slot_info(lot_id: str) -> List[dict]:
    """Slot 정보 조회 Mock"""
    print(f"  [MOCK] call_slot_info('{lot_id}') → {len(SAMPLE_SLOT)}건")
    return SAMPLE_SLOT

def call_lothis(lot_id: str) -> List[dict]:
    """LOT 이력 조회 Mock"""
    print(f"  [MOCK] call_lothis('{lot_id}') → {len(SAMPLE_LOTHIS)}건")
    return SAMPLE_LOTHIS

def call_eq(eqp_id: str) -> List[dict]:
    """장비 단건 조회 Mock"""
    all_eqp = SAMPLE_EQP_M15 + SAMPLE_EQP_M16
    result = [d for d in all_eqp if d["EQP_ID"] == eqp_id.upper()]
    print(f"  [MOCK] call_eq('{eqp_id}') → {len(result)}건")
    return result

def call_eqpm15(eqp_id: str = "*") -> List[dict]:
    """M15 장비 조회 Mock"""
    if eqp_id == "*":
        return SAMPLE_EQP_M15
    return [d for d in SAMPLE_EQP_M15 if d["EQP_ID"] == eqp_id]

def call_eqpm16(eqp_id: str = "*") -> List[dict]:
    """M16 장비 조회 Mock"""
    if eqp_id == "*":
        return SAMPLE_EQP_M16
    return [d for d in SAMPLE_EQP_M16 if d["EQP_ID"] == eqp_id]

def call_eqpm10(eqp_id: str = "*") -> List[dict]: return []
def call_eqpm11(eqp_id: str = "*") -> List[dict]: return []
def call_eqpm14(eqp_id: str = "*") -> List[dict]: return []

def call_operhis_api(lot_cd: str) -> List[dict]:
    """공정 이력 조회 Mock"""
    print(f"  [MOCK] call_operhis('{lot_cd}') → {len(SAMPLE_OPERHIS)}건")
    return SAMPLE_OPERHIS

print("Mock API 정의 완료")

## 4. GraphState 정의

In [ ]:
class GraphState(BaseModel):
    # 기본 입출력
    query: str = ""
    answer: Optional[str] = None
    query_type: Optional[str] = None
    intent: Optional[str] = None
    user_id: Optional[str] = None
    session_id: Optional[str] = None

    # 문서 검색
    chat_upload_retrieved_docs: Optional[List] = Field(default_factory=list)
    retrieved_docs: Optional[List] = Field(default_factory=list)

    # LOT / 장비 판별
    is_eqp: bool = False
    is_lot: bool = False
    eqp_id: Optional[str] = None
    eqp_data: Optional[List[dict]] = None
    lot_id: Optional[str] = None

    # FAB 관련
    fab: Optional[str] = None
    section_grp_nm: Optional[str] = None
    fab_model: Optional[str] = None
    fab_proc_model: Optional[str] = None
    fab_vendor: Optional[str] = None
    fab_area: Optional[str] = None
    fab_keyword: Optional[List[str]] = Field(default_factory=list)
    fab_group_by: Optional[str] = None
    fab_aggregate: Optional[Literal["count", "list"]] = "list"
    fab_chamber_only: bool = False

    # 공정 이력
    parsed_ctn_desc: Optional[str] = None
    parsed_lot_cd: Optional[str] = None
    parsed_is_prev: bool = False
    sql_result: Optional[List[dict]] = None
    skip_rag: bool = False

    class Config:
        arbitrary_types_allowed = True


# 테스트
state = GraphState(query="AB123456 슬롯 정보")
print(f"GraphState 생성: query='{state.query}', skip_rag={state.skip_rag}")

## 5. 헬퍼 함수들

### 5-1. extract_candidate — LOT/장비 ID 추출

In [ ]:
def extract_candidate(message: str) -> Optional[str]:
    """질문에서 LOT/장비 ID 후보를 정규식으로 추출"""
    patterns = [
        r'\b([A-Z]{2}\d{6,})\b',
        r'\b([A-Z]\d{7,})\b',
        r'\b([A-Z]{2}\d{4})\b',
        r'\b([A-Z]+[A-Z_0-9]+)\b',
        r'\b(CMP[A-Z0-9]+)\b',
        r'\b(KCC[A-Z0-9]+)\b',
        r'\b(\d[A-Z0-9]{3,7})\b',
        r'\b([A-Z]{3,}\d{2,})\b',
        r'\b([A-Z0-9]{3,})호기',
    ]
    for p in patterns:
        match = re.search(p, message.upper())
        if match:
            candidate = match.group(1).replace('호기', '')
            return candidate
    return None


# ──────────────── 테스트 ────────────────
test_cases = [
    "AB123456 슬롯 정보 알려줘",
    "M15A001 장비 상태",
    "M15 LPCVD 현황",
    "CVD 공정 어떻게 돼?",
    "3호기 상태 알려줘",
]

print("extract_candidate 테스트:")
for q in test_cases:
    result = extract_candidate(q)
    print(f"  '{q}' → {result}")

### 5-2. extract_fab — FAB 추출

In [ ]:
def extract_fab(message: str) -> Optional[str]:
    """M10~M16 FAB 코드 추출"""
    match = re.search(r'\b(M1[0-6])\b', message.upper())
    return match.group(1) if match else None


# ──────────────── 테스트 ────────────────
fab_test_cases = [
    "M15 장비 현황",
    "M16에서 ALD 장비",
    "M15X DRAM 공정",
    "장비 현황 알려줘",
]

print("extract_fab 테스트:")
for q in fab_test_cases:
    result = extract_fab(q)
    print(f"  '{q}' → {result}")

### 5-3. classify_equipment — 장비/챔버 분리

In [ ]:
def classify_equipment(data: List[Dict]) -> tuple:
    """EQP_ID 기준으로 메인 장비와 챔버를 분리"""
    main_eqp, chamber_eqp = [], []
    for item in data:
        eqp_id = item.get("EQP_ID", "")
        is_chamber = "_" in eqp_id or any(s in eqp_id for s in ["CH", "SPIN"])
        (chamber_eqp if is_chamber else main_eqp).append(item)
    return main_eqp, chamber_eqp


# ──────────────── 테스트 ────────────────
main_eqp, chamber_eqp = classify_equipment(SAMPLE_EQP_M15)
print(f"classify_equipment(M15 전체 {len(SAMPLE_EQP_M15)}건):")
print(f"  메인 장비 {len(main_eqp)}건: {[d['EQP_ID'] for d in main_eqp]}")
print(f"  챔버     {len(chamber_eqp)}건: {[d['EQP_ID'] for d in chamber_eqp]}")

### 5-4. get_eqp_by_fab — FAB 기준 장비 조회

In [ ]:
@functools.lru_cache(maxsize=32)
def get_eqp_by_fab(fab_input: str, eqp_id: str = "*", section_grp: str = None) -> List[Dict]:
    """FAB 코드로 장비 목록 조회. eqp_id='*'이면 전체 조회."""
    fab_key = fab_input.upper().replace('X', '')
    fab_api_map = {
        'M10': call_eqpm10, 'M11': call_eqpm11, 'M14': call_eqpm14,
        'M15': call_eqpm15, 'M16': call_eqpm16,
    }
    if fab_key not in fab_api_map:
        print(f"[WARN] 유효하지 않은 FAB: {fab_input}")
        return []

    raw_result = fab_api_map[fab_key](eqp_id) or []

    if section_grp:
        raw_result = [d for d in raw_result if d.get('SECTION_GRP_NM') == section_grp]

    return raw_result


# ──────────────── 테스트 ────────────────
m15_data = get_eqp_by_fab("M15", "*")
print(f"get_eqp_by_fab('M15', '*') → {len(m15_data)}건")

m15_nand = get_eqp_by_fab("M15", "*", "M15NAND")
print(f"get_eqp_by_fab('M15', '*', 'M15NAND') → {len(m15_nand)}건")

specific = get_eqp_by_fab("M15", "M15A001")
print(f"get_eqp_by_fab('M15', 'M15A001') → {specific}")

### 5-5. analyze_eqp_data — 장비 데이터 분석

In [ ]:
def analyze_eqp_data(data: List[Dict], eqp_id: str) -> Dict[str, Any]:
    """메인 장비와 챔버를 분리하고, PORT_INFO를 파싱하여 요약 딕셔너리 반환"""
    if not data:
        return {}

    main_eqp, chambers = None, []
    for d in data:
        cid = d.get("EQP_ID", "")
        if cid == eqp_id or (eqp_id in cid and "_" not in cid):
            main_eqp = d
        elif "_" in cid or cid.startswith(eqp_id):
            chambers.append(d)

    if not main_eqp and data:
        main_eqp = data[0]

    down_chambers = [c for c in chambers if c.get("MES_STAT_TYP") == "Down"]

    port_list = []
    port_info = main_eqp.get("PORT_INFO", "") if main_eqp else ""
    if port_info and port_info not in ("-", "", "N/A"):
        for segment in port_info.split(","):
            parts = segment.strip().split("l")
            if len(parts) >= 2:
                port_list.append({
                    "port": parts[0],
                    "transfer": parts[1] if len(parts) > 1 else "-",
                    "status": parts[2] if len(parts) > 2 else "-"
                })

    return {
        "EQP_ID": main_eqp.get("EQP_ID", eqp_id) if main_eqp else eqp_id,
        "EQ_GROUP": main_eqp.get("EQ_GROUP", "-") if main_eqp else "-",
        "SDPT_NM": main_eqp.get("SDPT_NM", "-") if main_eqp else "-",
        "FAB": main_eqp.get("FAC_ID", "-") if main_eqp else "-",
        "DFAB": main_eqp.get("DET_FAC_ID", "-") if main_eqp else "-",
        "장비사": main_eqp.get("VENDOR_NM", "-") if main_eqp else "-",
        "모델": main_eqp.get("EQP_MODEL_CD", "-") if main_eqp else "-",
        "BAY": main_eqp.get("BAY_NM", "-") if main_eqp else "-",
        "MES_STAT_TYP": main_eqp.get("MES_STAT_TYP", "-") if main_eqp else "-",
        "EQP_STAT_CD": main_eqp.get("EQP_STAT_CD", "-") if main_eqp else "-",
        "CHAMBER 수": len(chambers),
        "DOWN 수": len(down_chambers),
        "chamber_details": chambers,
        "port_list": port_list,
    }


# ──────────────── 테스트 ────────────────
eqp_data = get_eqp_by_fab("M15", "M15A001")
# 챔버도 포함하기 위해 전체 조회 후 M15A001 관련 데이터만 필터링
all_m15 = get_eqp_by_fab("M15", "*")
m15a001_group = [d for d in all_m15 if d["EQP_ID"].startswith("M15A001")]

analyzed = analyze_eqp_data(m15a001_group, "M15A001")
print("analyze_eqp_data 결과:")
for k, v in analyzed.items():
    if k not in ("chamber_details", "port_list"):
        print(f"  {k}: {v}")
print(f"  port_list: {analyzed['port_list']}")

## 6. 포매팅 함수들

### 6-1. format_eqp_status_table / format_eqp_info_table

In [ ]:
def format_eqp_info_table(analyzed: Dict, show_chamber: bool = True, show_port: bool = True) -> str:
    lines = ["## 장비 상태 \n"]
    lines += [" | 항목 | 값 |", " | ------ | ----- |"]
    for key in ["EQP_ID", "EQ_GROUP", "MES_STAT_TYP", "EQP_STAT_CD", "CHAMBER 수", "DOWN 수"]:
        lines.append(f" | {key} | {analyzed.get(key, '-')} |")

    if show_chamber and analyzed.get("chamber_details"):
        lines.append("\n | 챔버 ID | 상태 | EQP_STAT | LAST_EVENT |")
        lines.append("| --------- | ------ | -------- | ----------------- |")
        for ch in sorted(analyzed["chamber_details"], key=lambda x: x.get('EQP_ID') or ''):
            lines.append(f"| {ch.get('EQP_ID','-')} | {ch.get('MES_STAT_TYP','-')} | {ch.get('EQP_STAT_CD','-')} | {ch.get('LAST_EVENT_TM','-')} |")

    if show_port and analyzed.get("port_list"):
        lines += ["\n### PORT 정보", "", " | PORT | Transfer 상태 | 현재 상태 |", "| ------ | ---------------- | --------- |"]
        for p in analyzed["port_list"]:
            lines.append(f"| {p['port']} | {p['transfer']} | {p['status']} |")
    return '\n'.join(lines)


def format_eqp_status_table(analyzed: Dict, show_chamber: bool = True, show_port: bool = True) -> str:
    return format_eqp_info_table(analyzed, show_chamber, show_port)


# ──────────────── 테스트 ────────────────
result = format_eqp_info_table(analyzed)
display(Markdown(result))

### 6-2. format_setmo_data

In [ ]:
def format_setmo_data(data: List[dict], lot_id: str, filter_act: str = None) -> str:
    if not data:
        return f"{lot_id} SETMO 정보 없음"
    if filter_act:
        data = [d for d in data if d.get('ACT_NM') == filter_act]
    if not data:
        return f"{lot_id} Setmonitoring 정보 없음"

    if filter_act == "SetMonitor":
        lines = [f"##{lot_id} - Setmonitor\n", ""]
        lines += ["| 공정 | Comment | 측정 Slot | 엔지니어 |", "| ----- | --------- | --------- | --------- |"]
        for item in data[:20]:
            lines.append(f"| {item.get('OPER_DESC','-')} | {item.get('ACT_DESC','-')} | {item.get('MEAS_SLOT_NM','-')} | {item.get('ENGR_USER_NM','-')} |")
    elif filter_act == "makeOnHold":
        lines = [f"##{lot_id} - Future Hold \n", ""]
        lines += ["| 공정 | Comment | 엔지니어 |", "| ----- | -------- | ---------- |"]
        for item in data[:20]:
            lines.append(f"| {item.get('OPER_DESC','-')} | {item.get('ACT_DESC','-')} | {item.get('ENGR_USER_NM','-')} |")
    else:
        lines = [f"##{lot_id} - ALL \n", ""]
        lines += ["| 공정 | Comment | 측정 Slot | 엔지니어 |", "| ----- | --------- | --------- | --------- |"]
        for item in data[:20]:
            lines.append(f"| {item.get('OPER_DESC','-')} | {item.get('ACT_DESC','-')} | {item.get('MEAS_SLOT_NM','-')} | {item.get('ENGR_USER_NM','-')} |")

    if len(data) > 20:
        lines.append(f"\n*외 {len(data)-20}건 생략*")
    return '\n'.join(lines)


# ──────────────── 테스트 ────────────────
setmo_data = call_setmo_check("AB123456")

print("=== SetMonitor ===")
display(Markdown(format_setmo_data(setmo_data, "AB123456", "SetMonitor")))

print("=== Future Hold ===")
display(Markdown(format_setmo_data(setmo_data, "AB123456", "makeOnHold")))

### 6-3. format_slot_data

In [ ]:
def format_slot_data(data: List[Dict[str, Any]], lot_id: str) -> str:
    if not data:
        return f"{lot_id} 슬롯 정보 없음"

    latest_slots: Dict[int, Dict[str, Any]] = {}
    for d in data:
        pos_val = d.get("POSITION_VAL")
        if pos_val is None:
            continue
        try:
            pos_int = int(pos_val)
        except (ValueError, TypeError):
            continue
        event_tm = d.get('EVENT_TM', '')
        if pos_int not in latest_slots or event_tm > latest_slots[pos_int].get('EVENT_TM', ''):
            latest_slots[pos_int] = d

    lines = [f"##{lot_id} - Slot 정보(최신 기준)\n", ""]
    lines += [" | Slot | WF_ID | 공정 (FAB) | 시간 |", " | :----: | :----- | :---------- | :-----: |"]
    for i in range(1, 26):
        if i in latest_slots:
            item = latest_slots[i]
            lines.append(f" | {i} | {item.get('WF_ID','-')} | {item.get('OPER_DESC','-')} ({item.get('FAB','-')}) | {item.get('EVENT_TM','-')[:19]} |")
        else:
            lines.append(f" | {i} | *Empty* | - | - |")
    return '\n'.join(lines)


# ──────────────── 테스트 ────────────────
slot_data = call_slot_info("AB123456")
display(Markdown(format_slot_data(slot_data, "AB123456")))

### 6-4. format_lothis_data

In [ ]:
def format_lothis_data(data: List[Dict[str, Any]], lot_id: str) -> str:
    if not data:
        return f"{lot_id} 이력정보 없음"

    sorted_data = sorted(data, key=lambda x: x.get('TIMEKEY', ''), reverse=True)
    recent = sorted_data[:10]

    lines = [f"##{lot_id} - 이력정보 (최근 10건)\n", ""]
    lines.append(" | 순번 | 시간(timekey) | 이벤트(event_cd) | 공정 ID (OPER_ID) | 공정명 (CTN_DESC) | 수량 (WF_QTY) | 제품 ID(PROD_ID) |")
    lines.append("| :----: | :-------------: | :--------------------: | :----------------------: | :--------------------------------: | :------------------------------: | :-----------------: |")

    for idx, item in enumerate(recent, 1):
        raw_time = str(item.get('TIMEKEY', '-'))
        if len(raw_time) >= 14 and raw_time.isdigit():
            formatted_time = f"{raw_time[:4]}-{raw_time[4:6]}-{raw_time[6:8]} {raw_time[8:10]}:{raw_time[10:12]}:{raw_time[12:14]}"
        else:
            formatted_time = raw_time
        wf_qty = item.get('WF_QTY')
        lines.append(
            f" | {idx} | {formatted_time} | {item.get('EVENT_CD','-') or '-'} "
            f"| {item.get('OPER_ID','-') or '-'} | {item.get('CTN_DESC','-') or '-'} "
            f"| {wf_qty if wf_qty is not None else '-'} | {item.get('PROD_ID','-') or '-'} |"
        )
    return '\n'.join(lines)


# ──────────────── 테스트 ────────────────
lothis_data = call_lothis("AB123456")
display(Markdown(format_lothis_data(lothis_data, "AB123456")))

### 6-5. format_operhis_data

In [ ]:
def get_operhis_data(ctn_desc: str, lot_cd: Optional[str] = None, prev: bool = False, limit: int = 10) -> List[Dict]:
    """공정 이력에서 특정 공정(ctn_desc)의 이력 또는 이전 공정 이력을 반환"""
    if not lot_cd:
        return []
    try:
        all_data = call_operhis_api(lot_cd) or []
    except Exception as e:
        print(f"[ERROR] {e}")
        return []

    target_ctn = ctn_desc.strip().upper()
    ctn_items = [d for d in all_data if str(d.get("OPERATIONDESC", "")).strip().upper() == target_ctn]

    if not ctn_items:
        return []

    if prev:
        valid_levels = [d.get("OPERLEVEL") for d in ctn_items if d.get("OPERLEVEL") is not None]
        if not valid_levels:
            return []
        target_level = max(valid_levels)
        prev_items = [
            d for d in all_data
            if d.get("LOT_ID") and str(d["LOT_ID"]).startswith(str(lot_cd))
            and d.get("OPERLEVEL") is not None and d["OPERLEVEL"] < target_level
        ]
        return sorted(prev_items, key=lambda x: x.get("OPERLEVEL", 0), reverse=True)[:limit]
    else:
        return sorted(ctn_items, key=lambda x: x.get("OPERLEVEL", 0), reverse=True)[:limit]


def format_operhis_data(data: List[Dict], ctn_desc: str) -> str:
    lot_groups = defaultdict(list)
    for item in data:
        if item.get("LOT_ID"):
            lot_groups[item["LOT_ID"]].append(item)

    selected = sorted(
        [max(items, key=lambda x: x.get("OPERLEVEL", 0)) for items in lot_groups.values()],
        key=lambda x: x.get("OPERLEVEL", 0), reverse=True
    )[:10]

    lines = [f"##{ctn_desc} 공정 이력\n", "",
             "| LOT_ID | WF_QTY | CTN_DESC | MES_PROC_STAT_CD | LAST_EVENT_TM | FLOW_ID |",
             "| -------- | -------- | ---------- | ------------------------- | --------------------- | --------------- |"]
    for item in selected:
        lines.append(
            f" | {item.get('LOT_ID','-')} | {item.get('WF_QTY','-')} | "
            f"{item.get('CTN_DESC','-')} | {item.get('MES_PROC_STAT_CD','-')} | "
            f"{item.get('LAST_EVENT_TM','-')} | {item.get('FLOW_ID','-')} |"
        )
    if len(lot_groups) > 10:
        lines.append(f"\n*외 {len(lot_groups)-10}건 LOT 생략*")
    return '\n'.join(lines)


# ──────────────── 테스트 ────────────────
oper_data = get_operhis_data("CVD 증착", lot_cd="AB1", prev=False)
print(f"get_operhis_data('CVD 증착', prev=False) → {len(oper_data)}건")
display(Markdown(format_operhis_data(oper_data, "CVD 증착")))

print()
prev_data = get_operhis_data("CVD 증착", lot_cd="AB1", prev=True)
print(f"get_operhis_data('CVD 증착', prev=True) → {len(prev_data)}건 (이전 공정)")
if prev_data:
    display(Markdown(format_operhis_data(prev_data, "CVD 증착 이전 공정")))

### 6-6. format_fab_model_result

In [ ]:
def apply_filters(data: List[Dict], query_info: Dict) -> List[Dict]:
    """모델/공정그룹/벤더/구역/섹션 기준으로 장비 목록 필터링"""
    filtered = data
    if query_info.get("model"):
        filtered = [d for d in filtered if d.get("EQP_MODEL_CD") == query_info["model"]]
    if query_info.get("proc_model"):
        filtered = [d for d in filtered if d.get("EQ_GROUP") == query_info["proc_model"]]
    if query_info.get("vendor"):
        filtered = [d for d in filtered if d.get("VENDOR_NM") == query_info["vendor"]]
    if query_info.get("area"):
        filtered = [d for d in filtered if d.get("MGMT_AREA_ID") == query_info["area"]]
    if query_info.get("section_grp_nm"):
        filtered = [d for d in filtered if d.get("SECTION_GRP_NM") == query_info["section_grp_nm"]]
    return filtered


def format_fab_model_result(data: List[Dict], query_info: Dict, query: str) -> str:
    fab = query_info["fab"]
    chamber_only = query_info.get("chamber_only", False)
    all_models = set(e.get("EQP_MODEL_CD") for e in data)
    specific_model = next((e.get("EQP_MODEL_CD") for e in data), None)

    main_eqp = [e for e in data if "_" not in e.get("EQP_ID", "") and not any(s in e.get("EQP_ID", "") for s in ["CH", "SPIN"])]
    chamber_eqp = [e for e in data if "_" in e.get("EQP_ID", "") or any(s in e.get("EQP_ID", "") for s in ["CH", "SPIN"])]

    if len(all_models) == 1 and specific_model and (main_eqp or chamber_only):
        main_total, main_up = len(main_eqp), sum(1 for e in main_eqp if e.get("MES_STAT_TYP") == "Up")
        main_down = main_total - main_up
        ch_total, ch_up = len(chamber_eqp), sum(1 for e in chamber_eqp if e.get("MES_STAT_TYP") == "Up")
        ch_down = ch_total - ch_up

        lines = [f"##{query}\n", "", f"### {specific_model} DOWN 상태", "",
                 " | MODEL | TOTAL | UP | DOWN |", " | ------------- | ------------- | ------- | ------- |",
                 f"| {specific_model} | {main_total} | {main_up} | {main_down} |"]
        if ch_total > 0:
            lines.append(f" | {specific_model}(CHAMBER) | {ch_total} | {ch_up} | {ch_down} |")

        main_down_eqp = [e for e in main_eqp if e.get("MES_STAT_TYP") == "Down"][:10]
        if main_down_eqp:
            lines += ["", "### MAIN 장비 DOWN (상위 10개)", "",
                      " | EQP_ID | 상태 | 마지막 이벤트 |", " | ----------- | ------- | ------------------ |"]
            for e in main_down_eqp:
                lines.append(f"| {e.get('EQP_ID')} | {e.get('EQP_STAT_CD')} | {e.get('LAST_EVENT_TM','')} |")
        return '\n'.join(lines)

    else:
        models = defaultdict(lambda: {"TOTAL": 0, "UP": 0, "DOWN": 0})
        groups = defaultdict(lambda: {"TOTAL": 0, "UP": 0, "DOWN": 0})
        eqp_for_stats = chamber_eqp if chamber_only else main_eqp
        for e in eqp_for_stats:
            model = e.get("EQP_MODEL_CD", "Unknown")
            group = e.get("EQ_GROUP", "Unknown")
            stat = e.get("MES_STAT_TYP")
            models[model]["TOTAL"] += 1
            groups[group]["TOTAL"] += 1
            if stat == "Up":
                models[model]["UP"] += 1
                groups[group]["UP"] += 1
            else:
                models[model]["DOWN"] += 1
                groups[group]["DOWN"] += 1

        lines = [f"##{query} \n", "", "### 장비 현황(MODEL 기준)", "",
                 " | MODEL | TOTAL | UP | DOWN |", " | ------- | ------- | ----- | ------- |"]
        for k, v in sorted(models.items()):
            lines.append(f"| {k} | {v['TOTAL']} | {v['UP']} | {v['DOWN']} |")
        lines += ["", "### 장비 현황(EQP_GROUP 기준)", "",
                  " | EQ_GROUP | TOTAL | UP | DOWN |", " | ------- | ------- | ----- | ------- |"]
        for k, v in sorted(groups.items()):
            lines.append(f"| {k} | {v['TOTAL']} | {v['UP']} | {v['DOWN']} |")
        return '\n'.join(lines)


# ──────────────── 테스트 ────────────────
all_m15 = get_eqp_by_fab("M15", "*")

# 단일 모델 필터링 테스트
query_info_single = {"fab": "M15", "model": "LPCVD_A", "proc_model": None,
                     "vendor": None, "area": None, "section_grp_nm": None, "chamber_only": False}
filtered = apply_filters(all_m15, query_info_single)
print(f"LPCVD_A 필터 → {len(filtered)}건")
display(Markdown(format_fab_model_result(filtered, query_info_single, "M15 LPCVD_A 현황")))

print()

# 전체 집계 테스트
query_info_all = {"fab": "M15", "model": None, "proc_model": None,
                  "vendor": None, "area": None, "section_grp_nm": None, "chamber_only": False}
display(Markdown(format_fab_model_result(all_m15, query_info_all, "M15 전체 장비 현황")))

## 7. 워크플로우 노드 함수들

### 7-1. parse_fab_model_query

In [ ]:
def parse_fab_model_query(query: str) -> Optional[Dict[str, Any]]:
    """FAB+MODEL 조합 쿼리를 파싱하여 필터 조건 딕셔너리 반환"""
    if not query or not query.strip():
        return None

    query_upper = query.upper().strip()
    fab, section_grp_nm = None, None

    if "M15X" in query_upper:
        fab, section_grp_nm = "M15", "M15DRAM"
    elif "M15NAND" in query_upper:
        fab, section_grp_nm = "M15", "M15NAND"
    elif "M15" in query_upper:
        fab, section_grp_nm = "M15", "M15NAND"
    else:
        fab = extract_fab(query_upper)

    if not fab:
        return None

    try:
        fab_data = get_eqp_by_fab(fab, "*")
        if not fab_data:
            return None
    except:
        return None

    all_models  = set(e.get("EQP_MODEL_CD", "") for e in fab_data if e.get("EQP_MODEL_CD"))
    all_groups  = set(e.get("EQ_GROUP", "") for e in fab_data if e.get("EQ_GROUP"))
    all_vendors = set(e.get("VENDOR_NM", "") for e in fab_data if e.get("VENDOR_NM"))

    words = sorted(set(re.findall(r'\b[A-Z][A-Z0-9_]*\b', query_upper)), key=len, reverse=True)
    target_model = target_proc_model = target_vendor = target_area = None

    for word in words:
        if word == fab:
            continue
        if not target_model and word in all_models:
            target_model = word
        elif not target_proc_model and word in all_groups:
            target_proc_model = word
        elif not target_vendor and word in all_vendors:
            target_vendor = word

    if "CLEAN" in query_upper: target_area = "CLEAN"
    if "CMP" in query_upper:   target_area = "CMP"

    has_chamber = any(k in query for k in ['CHAMBER', '챔버'])
    has_summary = any(k in query for k in ['현황', '총', '댓수', '개수', '상태', 'DOWN', '다운'])
    group_by = None
    if '장비기준' in query or '모델별' in query: group_by = "model"
    elif '공정기준' in query or 'EQ_GROUP' in query: group_by = "eq_group"

    matched = {fab, target_model, target_proc_model, target_vendor, target_area}
    keywords = [w for w in words if w not in matched]

    return {
        "fab": fab, "model": target_model, "proc_model": target_proc_model,
        "vendor": target_vendor, "area": target_area, "section_grp_nm": section_grp_nm,
        "status_filter": None, "keywords": keywords, "chamber_only": has_chamber,
        "group_by": group_by, "aggregate": "count" if has_summary else "list",
        "search_all_fab": False,
    } if (fab or keywords) else None


# ──────────────── 테스트 ────────────────
fab_query_tests = [
    "M15 LPCVD_A 현황",
    "M15 CMP 장비 상태",
    "M16 ALD 장비 목록",
    "M15 전체 장비 현황",
]

print("parse_fab_model_query 테스트:")
for q in fab_query_tests:
    result = parse_fab_model_query(q)
    print(f"  '{q}'")
    if result:
        print(f"    fab={result['fab']}, model={result['model']}, group={result['proc_model']}, area={result['area']}")
    else:
        print(f"    → None")

### 7-2. parse_operation_query

In [ ]:
def parse_operation_query(query: str) -> Optional[Dict[str, Any]]:
    """공정 이력 질문에서 lot_cd, ctn_desc, is_prev를 추출"""
    if not query:
        return None

    query_upper = query.upper().strip()
    lot_cd = None
    match_lot = re.match(r"^([A-Z0-9]{3})", query_upper)
    if match_lot:
        lot_cd = match_lot.group(1)
        query_upper = query_upper[3:].strip()

    operhis_data = call_operhis_api(lot_cd)
    ctn_desc = set(item.get("OPERATIONDESC", "") for item in operhis_data if item.get("OPERATIONDESC"))

    matches = [ctn for ctn in sorted(ctn_desc, key=len, reverse=True) if ctn and ctn in query_upper]

    if not lot_cd and not ctn_desc:
        return None

    prev_keywords = ["전에", "이전", "before", "prior", "previous", "앞", "이전 공정", "이전단계", "전", "전 공정"]
    is_prev = any(kw in query for kw in prev_keywords) or any(kw in query.lower() for kw in prev_keywords)

    return {
        "lot_cd": lot_cd,
        "ctn_desc_candidate": matches[0] if matches else None,
        "is_prev": is_prev
    }


# ──────────────── 테스트 ────────────────
oper_tests = [
    "AB1 CVD 증착 공정 이력",
    "AB1 CVD 증착 이전 공정",
    "CVD 증착 현황",
]

print("parse_operation_query 테스트:")
for q in oper_tests:
    result = parse_operation_query(q)
    print(f"  '{q}' → {result}")

## 8. classify_and_execute 노드 (핵심)

질문을 분류하고 적절한 API를 호출하는 메인 노드입니다.

In [ ]:
def classify_and_execute(state: GraphState) -> GraphState:
    query = state.query
    query_lower = query.lower()
    query_upper = query.upper()

    print(f"[입력 질문] {query}")
    print("-" * 50)

    candidate = extract_candidate(query)
    print(f"[1단계] 후보 추출: {candidate}")

    # ── LOT 판별 ──────────────────────────────────
    is_lot = False
    if candidate:
        lot_data = call_lotid(candidate)
        if lot_data:
            is_lot = True
            state.is_lot = True
            state.lot_id = candidate
            print(f"[1단계] LOT 확인됨: {candidate}")

    if is_lot:
        print("[2단계] LOT → 세부 키워드 판별")

        setmo_keywords = ['셋모', 'setmo', 'setmonitor', '계측', 's/m']
        hold_keywords  = ['f/h', 'future hold', '퓨처홀드', 'fh']
        slot_keywords  = ['슬롯', 'slot', '정보', '상태', '어디', '위치', '공정']
        his_keywords   = ['이력']

        if any(k in query_lower for k in setmo_keywords):
            print("  → SETMO Monitor")
            state.intent = "setmonitoring"
            data = call_setmo_check(state.lot_id)
            state.answer = format_setmo_data(data, state.lot_id, "SetMonitor") if data else f"{state.lot_id} setmonitor 데이터 없음"
            state.skip_rag = True
            return state

        if any(k in query_lower for k in hold_keywords):
            print("  → Future Hold")
            state.intent = "future_action"
            data = call_setmo_check(state.lot_id)
            state.answer = format_setmo_data(data, state.lot_id, "makeOnHold") if data else f"{state.lot_id} Future Hold 없음"
            state.skip_rag = True
            return state

        if any(k in query_lower for k in slot_keywords):
            print("  → Slot Info")
            state.intent = "lot_info"
            data = call_slot_info(state.lot_id)
            state.answer = format_slot_data(data, state.lot_id) if data else f"{state.lot_id} 슬롯 정보 없음"
            state.skip_rag = True
            return state

        print("  → 기본 LOT 이력 조회")
        state.intent = "lot_his"
        data = call_lothis(state.lot_id)
        state.answer = format_lothis_data(data, state.lot_id) if data else f"{state.lot_id} 이력 없음"
        state.skip_rag = True
        return state

    # ── 장비 판별 ──────────────────────────────────
    print("[2단계] LOT 아님 → 장비 판별")
    eqp_candidate = candidate

    if eqp_candidate:
        try:
            eq_check = call_eq(eqp_candidate)
        except Exception as e:
            print(f"  call_eq 실패: {e}")
            eq_check = None

        if eq_check:
            actual_fab = eq_check[0].get("FAC_ID") or eq_check[0].get("SRC")
            print(f"  → 장비 확인됨: {eqp_candidate} ({actual_fab})")
            eqp_data = get_eqp_by_fab(actual_fab, eqp_candidate)
            all_fab_data = get_eqp_by_fab(actual_fab, "*")
            eqp_group = [d for d in all_fab_data if d["EQP_ID"].startswith(eqp_candidate)]

            if eqp_group:
                state.is_eqp = True
                state.eqp_id = eqp_candidate
                state.fab = actual_fab

                is_cmp = any("CMP" in str(d.get("MGMT_AREA_ID", "")).upper() for d in eqp_group)
                analyzed = analyze_eqp_data(eqp_group, eqp_candidate)

                info_keywords   = ['장비정보', '장비 정보', 'equipment info', '정보']
                status_keywords = ['장비상태', '장비 상태', '상태', 'down', 'up', '다운']

                if any(k in query_lower for k in info_keywords):
                    state.intent = "eqp_info"
                    state.answer = format_eqp_info_table(analyzed)
                else:
                    state.intent = "eqp_status"
                    state.answer = format_eqp_status_table(analyzed, show_chamber=not is_cmp, show_port=True)

                state.query_type = "sql_api"
                state.skip_rag = True
                return state

    # ── FAB+MODEL 판별 ──────────────────────────────
    print("[3단계] FAB+MODEL 판별")
    fab_model_query = parse_fab_model_query(query)

    if fab_model_query:
        fab = fab_model_query["fab"]
        raw_data = get_eqp_by_fab(fab, "*")
        filtered_data = apply_filters(raw_data, fab_model_query)
        filtered_data.sort(key=lambda x: x.get('EQP_ID', ''))

        state.fab = fab
        state.query_type = "sql_api"
        state.intent = "fab_model_query"
        state.skip_rag = True
        state.answer = format_fab_model_result(filtered_data, fab_model_query, query) if filtered_data else f"{fab} 조건에 맞는 장비 없음"
        return state

    # ── 공정 이력 판별 ──────────────────────────────
    print("[3.5단계] 공정 이력 판별")
    oper_info = parse_operation_query(query)

    if oper_info:
        oper_data = get_operhis_data(
            ctn_desc=oper_info.get("ctn_desc_candidate") or "",
            lot_cd=oper_info.get("lot_cd"),
            prev=oper_info.get("is_prev", False)
        )
        if oper_data:
            state.intent = "oper_step"
            state.query_type = "sql_api"
            state.skip_rag = True
            state.answer = format_operhis_data(oper_data, oper_info.get("ctn_desc_candidate") or "")
            return state

    # ── 일반 질문 (RAG) ──────────────────────────────
    print("[4단계] 일반 질문 → RAG")
    state.query_type = "general"
    state.intent = "general_qa"
    state.skip_rag = False
    return state


print("classify_and_execute 정의 완료")

## 9. 전체 워크플로우 시나리오 테스트

각 분류 경로별로 테스트합니다.

### 시나리오 1: LOT Slot 조회

In [ ]:
state = GraphState(query="AB123456 슬롯 정보 알려줘")
result = classify_and_execute(state)
print(f"\nintent: {result.intent} | skip_rag: {result.skip_rag}")
print()
display(Markdown(result.answer))

### 시나리오 2: LOT SetMonitor 조회

In [ ]:
state = GraphState(query="AB123456 setmo 확인해줘")
result = classify_and_execute(state)
print(f"\nintent: {result.intent} | skip_rag: {result.skip_rag}")
print()
display(Markdown(result.answer))

### 시나리오 3: LOT Future Hold 조회

In [ ]:
state = GraphState(query="AB123456 future hold 있어?")
result = classify_and_execute(state)
print(f"\nintent: {result.intent} | skip_rag: {result.skip_rag}")
print()
display(Markdown(result.answer))

### 시나리오 4: 장비 상태 조회

In [ ]:
state = GraphState(query="M15A001 장비 상태 알려줘")
result = classify_and_execute(state)
print(f"\nintent: {result.intent} | skip_rag: {result.skip_rag}")
print()
display(Markdown(result.answer))

### 시나리오 5: FAB+MODEL 집계 조회

In [ ]:
state = GraphState(query="M15 전체 장비 현황")
result = classify_and_execute(state)
print(f"\nintent: {result.intent} | skip_rag: {result.skip_rag}")
print()
display(Markdown(result.answer))

### 시나리오 6: 공정 이력 조회

In [ ]:
state = GraphState(query="AB1 CVD 증착 공정 이력")
result = classify_and_execute(state)
print(f"\nintent: {result.intent} | skip_rag: {result.skip_rag}")
print()
if result.answer:
    display(Markdown(result.answer))
else:
    print("(RAG 단계로 진행)")

### 시나리오 7: 일반 질문 (RAG fallback)

In [ ]:
state = GraphState(query="CVD 공정에서 두께 편차가 생기는 원인은?")
result = classify_and_execute(state)
print(f"intent: {result.intent} | skip_rag: {result.skip_rag}")
print("→ skip_rag=False이므로 conditional_retrieve → generate_response로 진행됩니다.")

## 10. 분류 결과 요약 테이블

In [ ]:
test_queries = [
    "AB123456 슬롯 정보 알려줘",
    "AB123456 setmo 확인해줘",
    "AB123456 future hold 있어?",
    "AB123456 이력 조회",
    "M15A001 장비 상태 알려줘",
    "M15A001 장비정보 알려줘",
    "M15 전체 장비 현황",
    "M15 LPCVD_A 현황",
    "M16 ALD 장비 현황",
    "AB1 CVD 증착 이전 공정",
    "CVD 공정 두께 편차 원인은?",
]

print("| 질문 | 분류 경로 | intent | skip_rag |")
print("|------|-----------|--------|----------|")
for q in test_queries:
    state = GraphState(query=q)
    # 출력 억제를 위한 wrapper
    import io, sys
    captured = io.StringIO()
    sys.stdout = captured
    result = classify_and_execute(state)
    sys.stdout = sys.__stdout__

    path = "LOT" if result.is_lot else ("EQP" if result.is_eqp else ("FAB/MODEL" if result.intent == "fab_model_query" else ("OPER" if result.intent == "oper_step" else "RAG")))
    print(f"| {q[:25]:<25} | {path:<10} | {result.intent or '-':<15} | {result.skip_rag} |")